In [ ]:
!pip install -q transformers datasets nltk torch pandas nump

ERROR: Could not find a version that satisfies the requirement nump (from versions: none)
ERROR: No matching distribution found for nump


In [ ]:
import os
import time
import json
import torch
import pandas as pd
import numpy as np
import nltk
from nltk.translate.gleu_score import sentence_gleu
from transformers import T5Tokenizer, T5ForConditionalGeneration
from google.colab import drive
# Ensure NLTK tokenizer models are downloaded
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
drive.mount('/content/drive')

# Load directly from Google Drive
MODEL_PATH = '/content/drive/MyDrive/t5_base_grammar_model'
TEST_DATA_PATH = '/content/test_holdout.csv'
RESULTS_OUTPUT_PATH = 'fp16_baseline_results.json'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running evaluation on device: {device}")

Mounted at /content/drive
Running evaluation on device: cpu


In [ ]:
# ----------------------------------------------------------------------
# 2. Load Fine-Tuned PyTorch FP16 Model & Tokenizer
# ----------------------------------------------------------------------
print(f"Loading baseline model from {MODEL_PATH}...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
model.eval()

Loading baseline model from /content/drive/MyDrive/t5_base_grammar_model...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
# ----------------------------------------------------------------------
# 3. Load Test Data
# ----------------------------------------------------------------------
print(f"Loading holdout test set from {TEST_DATA_PATH}...")
test_df = pd.read_csv(TEST_DATA_PATH)

# Use a representative sample of 500 rows for benchmarking latency & accuracy
EVAL_SAMPLE_SIZE = min(500, len(test_df))
eval_sample = test_df.sample(n=EVAL_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Evaluating on {EVAL_SAMPLE_SIZE} test sentences...")

Loading holdout test set from /content/test_holdout.csv...
Evaluating on 500 test sentences...


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
from nltk.translate.gleu_score import corpus_gleu

# ----------------------------------------------------------------------
# 4. Benchmarking Loop (Latency & GLEU Accuracy)
# ----------------------------------------------------------------------
print("\n" + "="*60)
print(" BENCHMARKING BASELINE PYTORCH MODEL (CPU) ")
print("="*60)

# Force CPU execution for a fair comparison against ONNX CPU
device = torch.device("cpu")
model = model.to(device).eval()

# Pre-tokenize inputs outside the timing block to measure pure model latency
print("Pre-tokenizing evaluation dataset...")
pretokenized_inputs = []
references = []

for idx, row in eval_sample.iterrows():
    input_text = str(row['input'])
    target_text = str(row['label'])

    encoded = tokenizer(input_text, return_tensors="pt", max_length=64, truncation=True)
    pretokenized_inputs.append({k: v.to(device) for k, v in encoded.items()})

    # Store reference tokens for corpus_gleu calculation
    references.append([target_text.split()])

# Warm-up run to clear CPU cold-start overhead
with torch.no_grad():
    _ = model.generate(**pretokenized_inputs[0], max_new_tokens=64, num_beams=1)

latencies = []
hypotheses = []

print(f"Running baseline CPU inference benchmark on {len(pretokenized_inputs)} samples...")

# Pure model generation benchmark
for inputs in pretokenized_inputs:
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, num_beams=1)
    t1 = time.perf_counter()

    latencies.append((t1 - t0) * 1000)

    # Decode and store hypotheses tokens for corpus_gleu
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    hypotheses.append(pred_text.split())


 BENCHMARKING BASELINE PYTORCH MODEL (CPU) 
Pre-tokenizing evaluation dataset...
Running baseline CPU inference benchmark on 500 samples...


In [ ]:
# ----------------------------------------------------------------------
# 5. Measure Model File Size
# ----------------------------------------------------------------------
def get_dir_size_mb(directory):
    total_size = 0
    for dirpath, _, filenames in os.walk(directory):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

model_size_mb = get_dir_size_mb(MODEL_PATH )

In [ ]:
# ----------------------------------------------------------------------
# 6. Calculate & Display Baseline Metrics
# ----------------------------------------------------------------------
# Calculate corpus-level GLEU score (references vs. hypotheses lists of tokens)
corpus_gleu_score = float(corpus_gleu(references, hypotheses))

mean_latency = float(np.mean(latencies))
p50_latency = float(np.percentile(latencies, 50))
p95_latency = float(np.percentile(latencies, 95))

results = {
    "model_name": "t5-base FP32 Baseline",
    "model_size_mb": round(model_size_mb, 2),
    "corpus_gleu_score": round(corpus_gleu_score, 4),
    "mean_latency_ms": round(mean_latency, 2),
    "p50_latency_ms": round(p50_latency, 2),
    "p95_latency_ms": round(p95_latency, 2),
    "sample_count": len(latencies)
}

print("\n" + "="*45)
print("       BASELINE FP32 EVALUATION RESULTS       ")
print("="*45)
print(f"Model Storage Size : {results['model_size_mb']} MB")
print(f"Corpus GLEU Score  : {results['corpus_gleu_score']}")
print(f"Mean CPU Latency   : {results['mean_latency_ms']} ms / sentence")
print(f"P50 CPU Latency    : {results['p50_latency_ms']} ms / sentence")
print(f"P95 CPU Latency    : {results['p95_latency_ms']} ms / sentence")
print("="*45)

# Save baseline metrics to JSON file
with open(RESULTS_OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=4)

print(f"\nSaved baseline evaluation results to: {RESULTS_OUTPUT_PATH}")


       BASELINE FP32 EVALUATION RESULTS       
Model Storage Size : 852.65 MB
Corpus GLEU Score  : 0.6351
Mean CPU Latency   : 2389.01 ms / sentence
P50 CPU Latency    : 1949.46 ms / sentence
P95 CPU Latency    : 4994.94 ms / sentence

Saved baseline evaluation results to: fp16_baseline_results.json
